# Aula 13 - Detecção de Anomalias

Quando o ponto estranho é o mais importante.

Na aula anterior, vimos que o DBSCAN marca alguns pontos como ruído (`label = -1`). Mas e se justamente esses pontos forem os mais importantes? Nesta prática, vamos explorar métodos de detecção de anomalias, desde técnicas univariadas simples até algoritmos multivariados.

## 1. Preparação

Execute as células em ordem. O notebook usa dados sintéticos para controlar a presença de anomalias conhecidas.

In [ ]:
# Pacotes necessários para executar este notebook:
# uv pip install numpy pandas matplotlib scikit-learn
#
# Nomes que podem confundir:
# - sklearn é instalado como scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

DRACULA = {
    'bg': '#282a36',
    'fg': '#f8f8f2',
    'cyan': '#8be9fd',
    'green': '#50fa7b',
    'pink': '#ff79c6',
    'red': '#ff5555',
    'yellow': '#f1fa8c',
    'purple': '#bd93f9',
    'muted': '#6272a4',
}

def apply_dracula(ax, title=None):
    ax.set_facecolor(DRACULA['bg'])
    ax.figure.set_facecolor(DRACULA['bg'])
    ax.tick_params(colors=DRACULA['fg'])
    ax.xaxis.label.set_color(DRACULA['fg'])
    ax.yaxis.label.set_color(DRACULA['fg'])
    ax.title.set_color(DRACULA['fg'])
    if title:
        ax.set_title(title, color=DRACULA['fg'], fontsize=14)
    return ax

## 2. Dataset sintético: comportamento de clientes

Vamos gerar dados de clientes com duas variáveis:

- `gasto_mensal`: valor médio gasto por mês
- `numero_compras`: quantidade de compras no período

A maioria dos clientes segue um padrão normal. Alguns pontos serão inseridos como anomalias deliberadas.

In [ ]:
np.random.seed(RANDOM_STATE)

n_normal = 300

normais = pd.DataFrame({
    "gasto_mensal": np.random.normal(500, 120, n_normal),
    "numero_compras": np.random.normal(12, 3, n_normal)
})

anomalias_reais = pd.DataFrame({
    "gasto_mensal": [1200, 1400, 100, 1600, 80],
    "numero_compras": [2, 3, 40, 1, 45]
})

dados = pd.concat([normais, anomalias_reais], ignore_index=True)
dados["anomalia_real"] = [0] * n_normal + [1] * len(anomalias_reais)

print(f"Total de pontos: {len(dados)}")
print(f"Pontos normais: {n_normal}")
print(f"Anomalias inseridas: {len(anomalias_reais)}")
print()
print(dados.describe())

## 3. Visualização inicial

Vamos plotar todos os pontos. As anomalias inseridas são conhecidas — podemos compará-las com o que os algoritmos detectarem.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

normais_plot = dados[dados["anomalia_real"] == 0]
anomalias_plot = dados[dados["anomalia_real"] == 1]

ax.scatter(normais_plot["gasto_mensal"], normais_plot["numero_compras"],
           c=DRACULA['cyan'], alpha=0.6, s=40, label="Normal")
ax.scatter(anomalias_plot["gasto_mensal"], anomalias_plot["numero_compras"],
           c=DRACULA['red'], marker="x", s=150, linewidths=3, label="Anomalia inserida")

ax.set_xlabel("Gasto mensal (R$)")
ax.set_ylabel("Número de compras")
ax.set_title("Clientes: comportamento normal e anômalo")
ax.legend()

apply_dracula(ax)
plt.tight_layout()
plt.show()

## 4. Método IQR (Interquartile Range)

O método IQR identifica outliers com base nos quartis de uma variável. Pontos fora de `Q1 - 1.5*IQR` ou `Q3 + 1.5*IQR` são considerados anômalos.

Vamos aplicar em cada variável separadamente e depois combinar os resultados.

In [ ]:
def detectar_iqr(serie):
    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1
    lim_inf = q1 - 1.5 * iqr
    lim_sup = q3 + 1.5 * iqr
    return (serie < lim_inf) | (serie > lim_sup)

outlier_gasto = detectar_iqr(dados["gasto_mensal"])
outlier_compras = detectar_iqr(dados["numero_compras"])

dados["anomalia_iqr"] = outlier_gasto | outlier_compras

n_iqr = dados["anomalia_iqr"].sum()
print(f"Anomalias detectadas pelo IQR: {n_iqr}")
print()

q1_g = dados["gasto_mensal"].quantile(0.25)
q3_g = dados["gasto_mensal"].quantile(0.75)
iqr_g = q3_g - q1_g
print(f"Gasto mensal: Q1={q1_g:.1f}, Q3={q3_g:.1f}, IQR={iqr_g:.1f}")
print(f"  Limites: [{q1_g - 1.5*iqr_g:.1f}, {q3_g + 1.5*iqr_g:.1f}]")

q1_c = dados["numero_compras"].quantile(0.25)
q3_c = dados["numero_compras"].quantile(0.75)
iqr_c = q3_c - q1_c
print(f"\nNúmero de compras: Q1={q1_c:.1f}, Q3={q3_c:.1f}, IQR={iqr_c:.1f}")
print(f"  Limites: [{q1_c - 1.5*iqr_c:.1f}, {q3_c + 1.5*iqr_c:.1f}]")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, var, label in zip(axes, ["gasto_mensal", "numero_compras"], ["Gasto mensal (R$)", "Número de compras"]):
    ax.boxplot(dados[var], patch_artist=True)
    ax.set_ylabel(label)
    apply_dracula(ax)

plt.suptitle("Boxplot: detecção de outliers por IQR", color=DRACULA['fg'], fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

normais_iqr = dados[~dados["anomalia_iqr"]]
anomalias_iqr = dados[dados["anomalia_iqr"]]

ax.scatter(normais_iqr["gasto_mensal"], normais_iqr["numero_compras"],
           c=DRACULA['cyan'], alpha=0.6, s=40, label="Normal")
ax.scatter(anomalias_iqr["gasto_mensal"], anomalias_iqr["numero_compras"],
           c=DRACULA['red'], marker="x", s=150, linewidths=3, label="Detectado pelo IQR")

ax.set_xlabel("Gasto mensal (R$)")
ax.set_ylabel("Número de compras")
ax.set_title(f"IQR: {n_iqr} anomalias detectadas")
ax.legend()

apply_dracula(ax)
plt.tight_layout()
plt.show()

## 5. DBSCAN como detector de anomalias

O DBSCAN marca pontos em regiões de baixa densidade como ruído (`label = -1`). Esses pontos são candidatos a anomalias.

Vamos padronizar os dados antes de aplicar o algoritmo, pois o DBSCAN usa distâncias.

In [ ]:
X = dados[["gasto_mensal", "numero_compras"]].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

modelo_dbscan = DBSCAN(eps=0.8, min_samples=10)
labels_dbscan = modelo_dbscan.fit_predict(X_scaled)

dados["label_dbscan"] = labels_dbscan
dados["anomalia_dbscan"] = labels_dbscan == -1

n_dbscan = dados["anomalia_dbscan"].sum()
n_clusters = len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)

print(f"Clusters encontrados: {n_clusters}")
print(f"Pontos marcados como ruído (anomalias): {n_dbscan}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

normais_db = dados[~dados["anomalia_dbscan"]]
anomalias_db = dados[dados["anomalia_dbscan"]]

ax.scatter(normais_db["gasto_mensal"], normais_db["numero_compras"],
           c=DRACULA['cyan'], alpha=0.6, s=40, label="Normal")
ax.scatter(anomalias_db["gasto_mensal"], anomalias_db["numero_compras"],
           c=DRACULA['red'], marker="x", s=150, linewidths=3, label="Ruído DBSCAN")

ax.set_xlabel("Gasto mensal (R$)")
ax.set_ylabel("Número de compras")
ax.set_title(f"DBSCAN: {n_dbscan} pontos marcados como ruído")
ax.legend()

apply_dracula(ax)
plt.tight_layout()
plt.show()

## 6. Isolation Forest

O Isolation Forest isola pontos com cortes aleatórios. Pontos anômalos são mais fáceis de isolar — precisam de menos cortes para serem separados.

O parâmetro `contamination` indica a proporção esperada de anomalias nos dados.

In [ ]:
modelo_if = IsolationForest(
    contamination=0.05,
    random_state=RANDOM_STATE
)

pred_if = modelo_if.fit_predict(X_scaled)
dados["anomalia_if"] = pred_if == -1

n_if = dados["anomalia_if"].sum()
print(f"Anomalias detectadas pelo Isolation Forest: {n_if}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

normais_if = dados[~dados["anomalia_if"]]
anomalias_if = dados[dados["anomalia_if"]]

ax.scatter(normais_if["gasto_mensal"], normais_if["numero_compras"],
           c=DRACULA['cyan'], alpha=0.6, s=40, label="Normal")
ax.scatter(anomalias_if["gasto_mensal"], anomalias_if["numero_compras"],
           c=DRACULA['red'], marker="x", s=150, linewidths=3, label="Isolation Forest")

ax.set_xlabel("Gasto mensal (R$)")
ax.set_ylabel("Número de compras")
ax.set_title(f"Isolation Forest: {n_if} anomalias detectadas")
ax.legend()

apply_dracula(ax)
plt.tight_layout()
plt.show()

## 7. Local Outlier Factor (LOF)

O LOF compara a densidade local de cada ponto com a densidade de seus vizinhos. Um ponto é suspeito se está em uma região muito menos densa que seus vizinhos.

Diferente do DBSCAN, o LOF não cria clusters — ele foca apenas na detecção de anomalias.

In [ ]:
modelo_lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.05
)

pred_lof = modelo_lof.fit_predict(X_scaled)
dados["anomalia_lof"] = pred_lof == -1

n_lof = dados["anomalia_lof"].sum()
print(f"Anomalias detectadas pelo LOF: {n_lof}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

normais_lof = dados[~dados["anomalia_lof"]]
anomalias_lof = dados[dados["anomalia_lof"]]

ax.scatter(normais_lof["gasto_mensal"], normais_lof["numero_compras"],
           c=DRACULA['cyan'], alpha=0.6, s=40, label="Normal")
ax.scatter(anomalias_lof["gasto_mensal"], anomalias_lof["numero_compras"],
           c=DRACULA['red'], marker="x", s=150, linewidths=3, label="LOF")

ax.set_xlabel("Gasto mensal (R$)")
ax.set_ylabel("Número de compras")
ax.set_title(f"LOF: {n_lof} anomalias detectadas")
ax.legend()

apply_dracula(ax)
plt.tight_layout()
plt.show()

## 8. Comparação dos métodos

Vamos comparar quantas anomalias cada método detectou e quantas das anomalias reais foram capturadas.

In [ ]:
def recall(real, prevista):
    verdadeiros_positivos = ((real == 1) & (prevista == True)).sum()
    total_reais = (real == 1).sum()
    return verdadeiros_positivos / total_reais if total_reais > 0 else 0

comparacao = pd.DataFrame({
    "metodo": ["IQR", "DBSCAN", "Isolation Forest", "LOF"],
    "anomalias_detectadas": [n_iqr, n_dbscan, n_if, n_lof],
    "recall_anomalias_reais": [
        recall(dados["anomalia_real"], dados["anomalia_iqr"]),
        recall(dados["anomalia_real"], dados["anomalia_dbscan"]),
        recall(dados["anomalia_real"], dados["anomalia_if"]),
        recall(dados["anomalia_real"], dados["anomalia_lof"]),
    ]
})

comparacao["recall_anomalias_reais"] = comparacao["recall_anomalias_reais"].apply(lambda x: f"{x:.0%}")

print(comparacao.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

colors = [DRACULA['cyan'], DRACULA['green'], DRACULA['purple'], DRACULA['yellow']]
bars = ax.barh(comparacao["metodo"], comparacao["anomalias_detectadas"], color=colors)

for bar, val in zip(bars, comparacao["anomalias_detectadas"]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            str(val), va='center', color=DRACULA['fg'], fontsize=12)

ax.set_xlabel("Número de anomalias detectadas")
ax.set_title("Comparação: anomalias detectadas por método")

apply_dracula(ax)
plt.tight_layout()
plt.show()

## 9. Discussão

Observe os resultados:

- Qual método detectou mais anomalias?
- Qual método capturou mais anomalias reais?
- Algum método marcou pontos normais como anomalia (falso positivo)?
- Algum método deixou passar anomalias reais (falso negativo)?

Não existe método perfeito. A escolha depende:

- do custo de falsos positivos vs. falsos negativos
- da natureza dos dados
- do contexto de negócio

### Pergunta para reflexão

Os pontos detectados parecem **erro**, **evento raro** ou **caso importante**? Justifique com base nos resultados.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metodos = [
    ("IQR", "anomalia_iqr"),
    ("DBSCAN", "anomalia_dbscan"),
    ("Isolation Forest", "anomalia_if"),
    ("LOF", "anomalia_lof"),
]

for ax, (nome, col) in zip(axes.flat, metodos):
    norm = dados[~dados[col]]
    anom = dados[dados[col]]
    
    ax.scatter(norm["gasto_mensal"], norm["numero_compras"],
               c=DRACULA['cyan'], alpha=0.5, s=30, label="Normal")
    ax.scatter(anom["gasto_mensal"], anom["numero_compras"],
               c=DRACULA['red'], marker="x", s=100, linewidths=2, label="Detectado")
    
    ax.set_xlabel("Gasto mensal (R$)")
    ax.set_ylabel("Número de compras")
    ax.set_title(f"{nome}: {anom.shape[0]} detectados")
    ax.legend(fontsize=9)
    apply_dracula(ax)

plt.suptitle("Comparação visual dos métodos", color=DRACULA['fg'], fontsize=16)
plt.tight_layout()
plt.show()